In [1]:
import torch
import torch.nn as nn

In [2]:
# input & output initialization
x = torch.tensor([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7],
    [1, 3, 5, 7, 9],
    [0, 2, 4, 6, 8],
    [4, 5, 6, 7, 8],
    [5, 4, 3, 2, 1],
    [6, 5, 4, 3, 2],
    [9, 7, 5, 3, 1],
    [3, 1, 4, 1, 5],
    [2, 7, 1, 8, 2],
    [4, 4, 4, 4, 4],
], dtype=torch.float32)
# 1 if increasing, 0 if decreasing (mixed or flat also count as 0)
y = torch.tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=torch.float32).view(-1, 1)
input_size = x.shape[-1]
output_size = y.shape[-1]
hidden_size = 4

In [3]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features=input_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=output_size),
        )

    def forward(self, input):
        return self.head(input=input)

In [4]:
# model initialization
model = MultiLayerPerceptron()
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.01)

head.0.weight: tensor([[-0.1751,  0.0974, -0.4426, -0.1934, -0.0613],
        [-0.1864,  0.0334,  0.4362,  0.0760,  0.4010],
        [ 0.4259, -0.2888, -0.1039, -0.4307,  0.1692],
        [-0.0560,  0.0835,  0.0037,  0.2570, -0.2774]])
head.0.bias: tensor([-0.2328,  0.1108,  0.4088,  0.3849])
head.2.weight: tensor([[ 0.2420, -0.1780, -0.1913, -0.3101]])
head.2.bias: tensor([-0.1261])


In [5]:
# training
epochs = 500
for epoch in range(epochs):
    # forward pass
    z = model(input=x)
    # calculate loss
    loss = nn.BCEWithLogitsLoss()(input=z, target=y)
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    if epoch % 50 == 0:
        print(f"epoch {epoch} loss: {loss.item()}")
        for name, param in model.named_parameters():
            print(f"{name}.grad: {param.grad}")
    # update parameters
    optimizer.step()

epoch 0 loss: 0.8650598526000977
head.0.weight.grad: tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0111,  0.0877,  0.2049,  0.3008,  0.4139],
        [-0.1165, -0.0873, -0.0795, -0.0459, -0.0424],
        [-0.0186,  0.0605,  0.2063,  0.2733,  0.4191]])
head.0.bias.grad: tensor([ 0.0000,  0.0396, -0.0200,  0.0364])
head.2.weight.grad: tensor([[ 0.0000, -1.6159,  0.0614,  0.0810]])
head.2.bias.grad: tensor([-0.2227])
epoch 50 loss: 0.21995426714420319
head.0.weight.grad: tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0098, -0.0422, -0.0671, -0.1012, -0.1224],
        [-0.0582, -0.0109, -0.0393,  0.0232, -0.0204],
        [-0.0238, -0.0272,  0.0198,  0.0072,  0.0541]])
head.0.bias.grad: tensor([ 0.0000, -0.0136, -0.0172, -0.0069])
head.2.weight.grad: tensor([[ 0.0000, -0.3789,  0.1048,  0.0642]])
head.2.bias.grad: tensor([-0.0847])
epoch 100 loss: 0.03948517516255379
head.0.weight.grad: tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
       

In [6]:
# prediction
with torch.inference_mode():
    z = model(input=x)
# activation function (value > 0 is 1, else 0)
y_pred = (z > 0).float()
print(f"y_pred: {y_pred.view(-1)}")
print(f"y:      {y.view(-1)}")

y_pred: tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
y:      tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
